In [1]:
from src.services.physics import PhysicsService
from scipy.integrate import solve_ivp

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_results(df, params_base):
    """
    Genera un gráfico dividido en dos paneles optimizado para Jupyter Notebooks.
    Sin guardado en disco para permitir interactividad en tiempo real (60 FPS).
    """
    # 1. Configurar el tema profesional de Seaborn
    sns.set_theme(
        style="whitegrid", 
        palette="deep",
        rc={
            "axes.spines.top": False,
            "axes.spines.right": False,
            "figure.facecolor": "white",
            "axes.facecolor": "white"
        }
    )
    
    # 2. Crear la figura (ligeramente más compacta para que quepa bien en el Notebook)
    fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(10, 8), sharex=True)
    fig.suptitle('Gemelo Digital: Dinámica Transitoria del CSTR', fontsize=16, fontweight='bold', y=0.98)

    color_c = '#2a75d3'
    color_t = '#d93838'

    # --- PANEL SUPERIOR: Concentración ---
    sns.lineplot(data=df, x='tau', y='Concentracion_C', ax=ax1, color=color_c, linewidth=2.5)
    ax1.fill_between(df['tau'], df['Concentracion_C'], alpha=0.1, color=color_c)
    
    ax1.set_ylabel(r"Concentración ($C'$)", fontsize=12, fontweight='bold', color=color_c)
    ax1.set_title("Decaimiento de la Concentración del Reactivo", fontsize=13, pad=10)
    ax1.set_xlabel('') 

    # Caja de parámetros interactiva
    texto_params = (
        "Parámetros Actuales:\n"
        "--------------------\n"
        f"Da_ref = {params_base['Da_ref']}\n"
        f"$\\gamma$    = {params_base['gamma']}\n"
        f"$\\beta$    = {params_base['beta']}\n"
        f"$\\kappa$    = {params_base['kappa']}"
    )
    props = dict(boxstyle='round,pad=0.5', facecolor='#f8f9fa', alpha=0.9, edgecolor='#ced4da')
    ax1.text(0.98, 0.95, texto_params, transform=ax1.transAxes, fontsize=10,
            verticalalignment='top', horizontalalignment='right', bbox=props, family='monospace')

    # --- PANEL INFERIOR: Temperatura ---
    sns.lineplot(data=df, x='tau', y='Temperatura_theta', ax=ax2, color=color_t, linewidth=2.5)
    ax2.fill_between(df['tau'], df['Temperatura_theta'], alpha=0.1, color=color_t)
    
    ax2.set_ylabel(r'Temperatura ($\theta$)', fontsize=12, fontweight='bold', color=color_t)
    ax2.set_xlabel(r'Tiempo Adimensional ($\tau$)', fontsize=12, fontweight='bold', labelpad=10)
    ax2.set_title("Exotermia y Respuesta del Sistema de Enfriamiento", fontsize=13, pad=10)

    # 3. Ajustes finales y renderizado
    plt.tight_layout()
    plt.subplots_adjust(top=0.88, hspace=0.25)

    plt.savefig("cstr_dynamics_split_hd.png", dpi=300, bbox_inches='tight')
    
    # Fundamental en Jupyter para que no se acumulen los gráficos antiguos en la memoria
    plt.show()
    plt.close(fig)

In [ ]:
import json
import ipywidgets as widgets
from IPython.display import display

# Importamos tus servicios (ajusta la ruta si es necesario)
from src.services.physics import PhysicsService
from src.services.simulation_service import SimulationService

# 1. Cargar la configuración base (solo para no empezar desde cero)
with open(r"src/common/config.json", 'r') as file:
    config_data = json.load(file)

sim_settings = config_data["Simulation_Settings"]
protecciones = config_data["Protections"]
condiciones_iniciales = config_data["Initial_Conditions"]["base_case"]

# 2. Definir la función interactiva con los Sliders
@widgets.interact(
    # Configuramos los rangos y valores por defecto de cada "perilla"
    Da_ref=widgets.FloatSlider(value=0.1, min=0.01, max=0.5, step=0.01, description='Da_ref (Velocidad):', style={'description_width': 'initial'}),
    gamma=widgets.FloatSlider(value=20.0, min=10.0, max=40.0, step=1.0, description='Gamma (Energía Act.):', style={'description_width': 'initial'}),
    beta=widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description='Beta (Exotermia):', style={'description_width': 'initial'}),
    kappa=widgets.FloatSlider(value=0.1, min=0.0, max=5.0, step=0.1, description='Kappa (Enfriamiento):', style={'description_width': 'initial'})
)
def simular_y_graficar_reactor(Da_ref, gamma, beta, kappa):
    """
    Esta función se ejecuta automáticamente cada vez que mueves un slider.
    """
    # 3. Inyectar los valores de las perillas a nuestra física
    fisica = PhysicsService(
        Da_ref=Da_ref, 
        beta=beta, 
        kappa=kappa, 
        gamma=gamma, 
        theta_0=0.5, 
        eps_divisor=protecciones.get("divisor", 1e-3)
    )
    
    # 4. Correr la simulación
    simulador = SimulationService(physics_service=fisica, simulation_settings=sim_settings)
    df_resultados = simulador.run_simulation(
        C_init=condiciones_iniciales["concentracion"], 
        theta_init=condiciones_iniciales["theta"]
    )
    
    # 5. Empaquetar los parámetros actuales para la caja de texto del gráfico
    params_actuales = {
        "Da_ref": round(Da_ref, 3), 
        "gamma": round(gamma, 1), 
        "beta": round(beta, 2), 
        "kappa": round(kappa, 1)
    }
    
    # 6. Llamar a la gráfica (se borrará y redibujará con cada movimiento)
    plot_results(df_resultados, params_actuales)

interactive(children=(FloatSlider(value=0.1, description='Da_ref (Velocidad):', max=0.5, min=0.01, step=0.01, …